In [4]:
import sys
from yourbench.pipeline.single_shot_question_generation import _load_stage_config
import random
from typing import Any
from dataclasses import field, dataclass

from loguru import logger

from datasets import Dataset
from yourbench.utils.prompts import (
    QUESTION_GENERATION_USER_PROMPT,
    QUESTION_GENERATION_SYSTEM_PROMPT,
    QUESTION_GENERATION_SYSTEM_PROMPT_MULTI,
)
from yourbench.utils.dataset_engine import (
    custom_load_dataset,
    custom_save_dataset,
)

# Import the unified parsing function
from yourbench.utils.parsing_engine import shuffle_mcq, parse_qa_pairs_from_response
from yourbench.utils.inference_engine import InferenceCall, run_inference
from yourbench.utils.loading_engine import load_config


/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
config=load_config("./my_example.yaml")

2025-09-29 14:00:48.016 | DEBUG    | yourbench.utils.loading_engine:load:42 - Read configuration from my_example.yaml
2025-09-29 14:00:48.029 | DEBUG    | yourbench.utils.loading_engine:load:46 - Configuration loaded successfully from my_example.yaml


In [3]:
config.keys()

dict_keys(['hf_configuration', 'language_of_interest', 'model_list', 'model_roles', 'pipeline'])

In [10]:

file_dir=config["pipeline"]["ingestion"]["source_documents_dir"]

In [15]:
import os
files=[f.split('.pdf')[0] for f in os.listdir(file_dir) if f.endswith(".pdf")]
files

['SwedenDrivingCourse_humanRisk']

In [17]:
f_name='_'.join(files) + '.pdf'
f_name

'SwedenDrivingCourse_humanRisk.pdf'

In [6]:
from datasets import Dataset, DatasetDict, load_dataset, load_from_disk, concatenate_datasets
stage_config = _load_stage_config(config)
chunked_dataset = load_from_disk(dataset_path="./example/data/local_saved/English/chunked")
summarized_dataset = load_from_disk(dataset_path="./example/data/local_saved/English/summarized")
single_shot_questions = load_from_disk(dataset_path="./example/data/local_saved/English/single_shot_questions")

In [5]:
local_dataset_dir = config["hf_configuration"].get("local_dataset_dir")
local_dataset_dir

'example/data/local_saved/English'

In [8]:
single_shot_qs=single_shot_questions.to_pandas()

In [14]:
single_shot_qs.to_csv("/workspace/example/data/csv/SwedenDrivingCourse_StartDriving.csv", index=False)

237

In [8]:
'yes'if local_dataset_dir else 'no'

'yes'

In [9]:
dataset=concatenate_datasets([chunked_dataset, summarized_dataset])

In [ ]:
n=len(dataset)

for row_index, row in zip(range(n), single_shot_questions):
    print("---"*20)
    print("row_index:", row_index, "\n", row.keys())
    print("chunks =\n", row["chunks"])
    print("--.---"*15)
    print("document_summary =\n", row["document_summary"])
    

In [10]:
len(dataset)

2

In [12]:
n=len(dataset)

for row_index, row in zip(range(n), dataset):
    print("---"*20)
    print("row_index:", row_index, "\n", row.keys())
    print("chunks =\n", row["chunks"])
    print("--.---"*15)
    print("document_summary =\n", row["document_summary"])
    

------------------------------------------------------------
row_index: 0 
 dict_keys(['document_id', 'document_text', 'document_filename', 'document_metadata', 'chunks', 'multihop_chunks', 'chunk_info_metrics', 'chunking_model', 'raw_chunk_summaries', 'chunk_summaries', 'raw_document_summary', 'document_summary', 'summarization_model'])
chunks =
 [{'chunk_id': 'e8b5b6b9-e15a-4e71-a09d-4de1e644a316_0', 'chunk_text': 'Before Driving – Private Supervisor  Supervisors for private practice driving for a category B license must have undergone an introductory course before they can be approved as supervisors. As a student, you must have gone through a valid introductory course, if you want to partake in private practice driving for category B. You must have completed the course in order for the supervisor to have their application processed by the Swedish Transport Agency. You, as a student, must be at least 15 years and 9 months old in order to attend the education. You may partake in priva

In [14]:
single_shot_dataset = load_from_disk(dataset_path="./example/data/local_saved/English/single_shot_questions")

In [18]:
from colorama import Fore
n=len(single_shot_dataset)

for row_index, row in zip(range(n), single_shot_dataset):
    print("---"*20)
    print("row_index:", row_index, "\n", row.keys())
    print(Fore.BLUE + "question =\n", row["question"],Fore.RESET)
    print("--.---"*15)
    print(Fore.CYAN +"choices =\n", row["choices"])
    print(Fore.GREEN+ "raw_response =\n", row["raw_response"],Fore.RESET)
    #print(Fore.GREEN+ "thought process =\n", row["thought_process"],Fore.RESET)
    print("self_answer =\n", row["self_answer"],Fore.RESET)
    print(Fore.LIGHTMAGENTA_EX +"citations =\n", row["citations"], Fore.RESET)
    

------------------------------------------------------------
row_index: 0 
 dict_keys(['chunk_id', 'document_id', 'additional_instructions', 'question', 'self_answer', 'choices', 'estimated_difficulty', 'self_assessed_question_type', 'generating_model', 'thought_process', 'raw_response', 'citations'])
question =
 What is the minimum age requirement for a private supervisor in a Swedish Driving License Course? 
--.-----.-----.-----.-----.-----.-----.-----.-----.-----.-----.-----.-----.-----.-----.---
choices =
 []
raw_response =
 ### <document_analysis>

**Text Chunk Analysis:**

The provided `<text_chunk>` outlines the eligibility criteria and regulations for becoming a private supervisor in a Swedish Driving License Course, specifically for category B vehicles. Key points identified include:

1. **Age Requirement**: Supervisor must be at least 24 years old.
2. **License Eligibility**: Valid EEA-issued driving license for the vehicle type, held for at least five of the last ten years.
